# FANTOM5 time course analysis
## Setup gene pairs TSSs
### Author: Martin Loza
### Date: 25/12/30

Since the FANTOM5 CTSS are in the hg19 genome, we will liftover the region around gene pairs TSSs to match those of the FANTOM5.

In [17]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(dplyr)
    library(ggvenn)
})

# Local variables 
seed = 777
date = "251230"

# Define colors for strand plots
red = "#E41A1C"
blue = "#377EB8"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
gray = "gray50"
text_size = 18
width = 18.6
height = 5
dot_size = 4
line_size = 1.5
dpi = 300

gene_pairs_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Supplementary/"
out_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/09_FANTOM5_time_course_analyses/Results/"
# Local Functions



### Load and setup the data

Load the expression data of selected gene pairs

In [18]:
# Load the gene pairs
gene_pairs_ranked = read.table(paste0(gene_pairs_dir, "Supplementary_table_ranked_lncRNA_TF_gene_pairs_251218.tsv"),
                        sep = "\t", header = TRUE, stringsAsFactors = FALSE)
head(gene_pairs_ranked)

,gene_pair_id,gene_name_pair_id,lncRNA_Family,avg_correlation,overall_rank,is_DT,is_AS
,<chr>,<chr>,<chr>,<dbl>,<int>,<lgl>,<lgl>
1,ENSG00000227640_ENSG00000125285,SOX21-AS1_SOX21,HMG,0.9682961,1,FALSE,TRUE
2,ENSG00000263146_ENSG00000256463,LINC01896_SALL3,zf-C2H2,0.9499586,2,FALSE,FALSE
3,ENSG00000277268_ENSG00000273706,LHX1-DT_LHX1,Homeobox,0.9465009,3,TRUE,FALSE
4,ENSG00000236502_ENSG00000138083,SIX3-AS1_SIX3,Homeobox,0.9460684,4,FALSE,TRUE
5,ENSG00000255399_ENSG00000089225,TBX5-AS1_TBX5,T-box,0.9433313,5,FALSE,TRUE
6,ENSG00000240990_ENSG00000005073,HOXA11-AS_HOXA11,Homeobox,0.9392763,6,FALSE,TRUE


Load the gene information to recover the TSS of gene pairs

In [19]:
gene_pairs_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/human_unique_lncRNA_TF_pairs_10000bp_251215.tsv"
gene_pairs_info <- read.table(gene_pairs_file, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
cat("Number of gene pairs: ", nrow(gene_pairs_info), "\n")
head(gene_pairs_info)

Number of gene pairs:  1978 


,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,⋯,Family,is_TF,abs_strand_distance,ncrna_gene_id,pcg_gene_id,gene_pair_id,gene_name_pair_id,ncrna_is_mane,tf_is_mane,mane_priority
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,⋯,<chr>,<lgl>,<int>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<int>
1,11,ENST00000381363,2140644,IGF2-AS,1,lncRNA,ENST00000643349,unnamed,2149603,8959,⋯,ZBTB,TRUE,8959,ENSG00000099869,ENSG00000284779,ENSG00000099869_ENSG00000284779,IGF2-AS_unnamed,FALSE,FALSE,4
2,11,ENST00000833483,61756482,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-1093,⋯,NDT80_PhoG,TRUE,1093,ENSG00000124915,ENSG00000124920,ENSG00000124915_ENSG00000124920,MYRF-AS1_MYRF,FALSE,FALSE,4
3,2,ENST00000440903,222302978,CCDC140,1,lncRNA,ENST00000392070,PAX3,222298998,-3980,⋯,PAX,TRUE,3980,ENSG00000163081,ENSG00000135903,ENSG00000163081_ENSG00000135903,CCDC140_PAX3,FALSE,TRUE,2
4,17,ENST00000738117,76562607,SNHG16,1,lncRNA,ENST00000713548,unnamed,76569648,7041,⋯,ZBTB,TRUE,7041,ENSG00000163597,ENSG00000284526,ENSG00000163597_ENSG00000284526,SNHG16_unnamed,FALSE,TRUE,2
5,5,ENST00000809704,136132805,SMAD5-AS1,-1,lncRNA,ENST00000545279,SMAD5,136132845,40,⋯,MH1,TRUE,40,ENSG00000164621,ENSG00000113658,ENSG00000164621_ENSG00000113658,SMAD5-AS1_SMAD5,FALSE,TRUE,2
6,10,ENST00000625168,45000920,ZNF22-AS1,-1,lncRNA,ENST00000298299,ZNF22,45000923,3,⋯,zf-C2H2,TRUE,3,ENSG00000165511,ENSG00000165512,ENSG00000165511_ENSG00000165512,ZNF22-AS1_ZNF22,FALSE,TRUE,2


Let's recover the genes' TSSs.

In [20]:
# Transfer TSS information to the ranked gene pairs table
gene_pairs_ranked <- gene_pairs_ranked %>%
    dplyr::left_join(gene_pairs_info %>%
                     select(gene_pair_id, ncrna_tss, pcg_tss, chromosome),
                        by = "gene_pair_id")
head(gene_pairs_ranked)

,gene_pair_id,gene_name_pair_id,lncRNA_Family,avg_correlation,overall_rank,is_DT,is_AS,ncrna_tss,pcg_tss,chromosome
,<chr>,<chr>,<chr>,<dbl>,<int>,<lgl>,<lgl>,<int>,<int>,<chr>
1,ENSG00000227640_ENSG00000125285,SOX21-AS1_SOX21,HMG,0.9682961,1,FALSE,TRUE,94712716,94712545,13
2,ENSG00000263146_ENSG00000256463,LINC01896_SALL3,zf-C2H2,0.9499586,2,FALSE,FALSE,78979074,78979818,18
3,ENSG00000277268_ENSG00000273706,LHX1-DT_LHX1,Homeobox,0.9465009,3,TRUE,FALSE,36936670,36936784,17
4,ENSG00000236502_ENSG00000138083,SIX3-AS1_SIX3,Homeobox,0.9460684,4,FALSE,TRUE,44941582,44941702,2
5,ENSG00000255399_ENSG00000089225,TBX5-AS1_TBX5,T-box,0.9433313,5,FALSE,TRUE,114408426,114408442,12
6,ENSG00000240990_ENSG00000005073,HOXA11-AS_HOXA11,Homeobox,0.9392763,6,FALSE,TRUE,27185235,27185232,7


### Setup TSS for lifover: hg38 -> hg19

In [21]:
# Let's create unique gene_tss ids for liftover
# The IDs are in the gene_pair_id column as lncRNAid_PCGid
gene_pairs_ranked <- gene_pairs_ranked %>%
    mutate(lncRNA_id = str_split_fixed(gene_pair_id, "_", 2)[,1],
           TF_id = str_split_fixed(gene_pair_id, "_", 2)[,2]) %>%
    mutate(lncRNA_tss_id = paste0(lncRNA_id, "_", ncrna_tss),
           TF_tss_id = paste0(TF_id, "_", pcg_tss))
head(gene_pairs_ranked)

,gene_pair_id,gene_name_pair_id,lncRNA_Family,avg_correlation,overall_rank,is_DT,is_AS,ncrna_tss,pcg_tss,chromosome,lncRNA_id,TF_id,lncRNA_tss_id,TF_tss_id
,<chr>,<chr>,<chr>,<dbl>,<int>,<lgl>,<lgl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ENSG00000227640_ENSG00000125285,SOX21-AS1_SOX21,HMG,0.9682961,1,FALSE,TRUE,94712716,94712545,13,ENSG00000227640,ENSG00000125285,ENSG00000227640_94712716,ENSG00000125285_94712545
2,ENSG00000263146_ENSG00000256463,LINC01896_SALL3,zf-C2H2,0.9499586,2,FALSE,FALSE,78979074,78979818,18,ENSG00000263146,ENSG00000256463,ENSG00000263146_78979074,ENSG00000256463_78979818
3,ENSG00000277268_ENSG00000273706,LHX1-DT_LHX1,Homeobox,0.9465009,3,TRUE,FALSE,36936670,36936784,17,ENSG00000277268,ENSG00000273706,ENSG00000277268_36936670,ENSG00000273706_36936784
4,ENSG00000236502_ENSG00000138083,SIX3-AS1_SIX3,Homeobox,0.9460684,4,FALSE,TRUE,44941582,44941702,2,ENSG00000236502,ENSG00000138083,ENSG00000236502_44941582,ENSG00000138083_44941702
5,ENSG00000255399_ENSG00000089225,TBX5-AS1_TBX5,T-box,0.9433313,5,FALSE,TRUE,114408426,114408442,12,ENSG00000255399,ENSG00000089225,ENSG00000255399_114408426,ENSG00000089225_114408442
6,ENSG00000240990_ENSG00000005073,HOXA11-AS_HOXA11,Homeobox,0.9392763,6,FALSE,TRUE,27185235,27185232,7,ENSG00000240990,ENSG00000005073,ENSG00000240990_27185235,ENSG00000005073_27185232


In [22]:
# Let's merge the TSS information into a single table for liftover
tss_table_hg38 <- rbind(gene_pairs_ranked %>%
                            select(lncRNA_tss_id, ncrna_tss, chromosome) %>%
                            rename(gene_tss_id = lncRNA_tss_id,
                                   gene_tss = ncrna_tss),
                        gene_pairs_ranked %>%
                            select(TF_tss_id, pcg_tss, chromosome) %>%
                            rename(gene_tss_id = TF_tss_id,
                                   gene_tss = pcg_tss)) %>%
    distinct()
cat("Number of unique TSS entries: ", nrow(tss_table_hg38), "\n")
head(tss_table_hg38,3)

Number of unique TSS entries:  506 


,gene_tss_id,gene_tss,chromosome
,<chr>,<int>,<chr>
1,ENSG00000227640_94712716,94712716,13
2,ENSG00000263146_78979074,78979074,18
3,ENSG00000277268_36936670,36936670,17


In [23]:
# Now, let's create a BED file for liftover and save it for analysis in the UCSC Genome Browser
# Let's create regions of 100bp centered on the TSS
tss_bed_hg38 <- tss_table_hg38 %>% 
    mutate(chromosome = paste0("chr", chromosome),
            start = gene_tss - 50,
           end = gene_tss + 50) %>%
    select(chromosome, start, end, gene_tss_id) %>%
    arrange(chromosome, start)
head(tss_bed_hg38,3)

# Save the BED file
write.table(tss_bed_hg38, file = paste0(out_dir, "gene_pairs_TSS_hg38.bed"), quote = FALSE, sep = "\t", row.names = FALSE, col.names = FALSE)


,chromosome,start,end,gene_tss_id
,<chr>,<dbl>,<dbl>,<chr>
1,chr1,919642,919742,ENSG00000223764_919692
2,chr1,923873,923973,ENSG00000187634_923923
3,chr1,2530014,2530114,ENSG00000272449_2530064


### Liftover in UCSC genome browser

Let's make the lifover using the webtool in UCSC genome browser

### Hg38 to hg19

Let's load the liftovered TSSs and transfer their information for mapping to FANTOM5 TSSs clusters

In [24]:
# Load the liftover results
tss_liftover_results <- read.table(paste0(out_dir, "gene_pairs_TSS_hg19.bed"),
                                   sep = "\t", header = FALSE, stringsAsFactors = FALSE)
colnames(tss_liftover_results) <- c("chromosome", "start", "end", "gene_tss_id")
head(tss_liftover_results,3)


,chromosome,start,end,gene_tss_id,NA
,<chr>,<int>,<int>,<chr>,<int>
1,chr1,855022,855122,ENSG00000223764_919692,1
2,chr1,859253,859353,ENSG00000187634_923923,1
3,chr1,2461453,2461553,ENSG00000272449_2530064,1


In [25]:
cat("Identical number of entries before and after liftover: ", nrow(tss_table_hg38) == nrow(tss_liftover_results), "\n")
cat("Any duplicated gene_tss_id after liftover: ", any(duplicated(tss_liftover_results$gene_tss_id)), "\n")

Identical number of entries before and after liftover:  TRUE 
Any duplicated gene_tss_id after liftover:  FALSE 


In [26]:
# Let's transfer the hg19 TSS positions back to the ranked gene pairs table
gene_pairs_ranked_hg19 <- gene_pairs_ranked %>%
    # Join lncRNA TSS
    dplyr::left_join(tss_liftover_results %>%
                        select(gene_tss_id, chromosome, start, end) %>%
                     rename(lncRNA_tss_id = gene_tss_id,
                            lncrna_tss_hg19_chromosome = chromosome,
                            lncrna_tss_hg19_start = start,
                            lncrna_tss_hg19_end = end),
                     by = "lncRNA_tss_id") %>%
    # Join TF TSS
    dplyr::left_join(tss_liftover_results %>%
                        select(gene_tss_id, chromosome, start, end) %>%
                     rename(TF_tss_id = gene_tss_id,
                            tf_tss_hg19_chromosome = chromosome,
                            tf_tss_hg19_start = start,
                            tf_tss_hg19_end = end),
                     by = "TF_tss_id")
head(gene_pairs_ranked_hg19)
    

,gene_pair_id,gene_name_pair_id,lncRNA_Family,avg_correlation,overall_rank,is_DT,is_AS,ncrna_tss,pcg_tss,chromosome,lncRNA_id,TF_id,lncRNA_tss_id,TF_tss_id,lncrna_tss_hg19_chromosome,lncrna_tss_hg19_start,lncrna_tss_hg19_end,tf_tss_hg19_chromosome,tf_tss_hg19_start,tf_tss_hg19_end
,<chr>,<chr>,<chr>,<dbl>,<int>,<lgl>,<lgl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<int>,<int>
1,ENSG00000227640_ENSG00000125285,SOX21-AS1_SOX21,HMG,0.9682961,1,FALSE,TRUE,94712716,94712545,13,ENSG00000227640,ENSG00000125285,ENSG00000227640_94712716,ENSG00000125285_94712545,chr13,95364920,95365020,chr13,95364749,95364849
2,ENSG00000263146_ENSG00000256463,LINC01896_SALL3,zf-C2H2,0.9499586,2,FALSE,FALSE,78979074,78979818,18,ENSG00000263146,ENSG00000256463,ENSG00000263146_78979074,ENSG00000256463_78979818,chr18,76739024,76739124,chr18,76739768,76739868
3,ENSG00000277268_ENSG00000273706,LHX1-DT_LHX1,Homeobox,0.9465009,3,TRUE,FALSE,36936670,36936784,17,ENSG00000277268,ENSG00000273706,ENSG00000277268_36936670,ENSG00000273706_36936784,chr17,35293919,35294019,chr17,35294033,35294133
4,ENSG00000236502_ENSG00000138083,SIX3-AS1_SIX3,Homeobox,0.9460684,4,FALSE,TRUE,44941582,44941702,2,ENSG00000236502,ENSG00000138083,ENSG00000236502_44941582,ENSG00000138083_44941702,chr2,45168671,45168771,chr2,45168791,45168891
5,ENSG00000255399_ENSG00000089225,TBX5-AS1_TBX5,T-box,0.9433313,5,FALSE,TRUE,114408426,114408442,12,ENSG00000255399,ENSG00000089225,ENSG00000255399_114408426,ENSG00000089225_114408442,chr12,114846181,114846281,chr12,114846197,114846297
6,ENSG00000240990_ENSG00000005073,HOXA11-AS_HOXA11,Homeobox,0.9392763,6,FALSE,TRUE,27185235,27185232,7,ENSG00000240990,ENSG00000005073,ENSG00000240990_27185235,ENSG00000005073_27185232,chr7,27224804,27224904,chr7,27224801,27224901


In [27]:
# Save the updated ranked gene pairs table with hg19 TSS information
write.table(gene_pairs_ranked_hg19,
            file = paste0(out_dir, "Ranked_lncRNA_TF_gene_pairs_hg19_TSS_", date, ".tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)